In [1]:
!pip install sentence-transformers chromadb transformers accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 89.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.3 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Fou

In [2]:
import pandas as pd

#Loading the csv
df = pd.read_csv("/kaggle/input/datasets/thedevastator/pubmed-article-summarization-dataset/train.csv")

#Drop any rows with abstract missing
df = df.dropna(subset = ['abstract'])
print(f"{df.count()}")
df.head(5)

article     117232
abstract    119924
dtype: int64


,article,abstract
0,a recent systematic analysis showed that in 20...,background : the present study was carried out...
1,it occurs in more than 50% of patients and may...,backgroundanemia in patients with cancer who a...
2,"tardive dystonia ( td ) , a rarer side effect ...",tardive dystonia ( td ) is a serious side effe...
3,"lepidoptera include agricultural pests that , ...",many lepidopteran insects are agricultural pes...
4,syncope is caused by transient diffuse cerebra...,we present an unusual case of recurrent cough ...


In [3]:
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
import chromadb

df_subset = df.head(1000).copy()
#cleaning the database
def simpleclean(text):
    if not isinstance(text, str):
        return ""
    text = text.lower().replace('\n',' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df_subset['abstract_cleaned'] = df['abstract'].apply(simpleclean)

#Load the model to convert the text into embeddings
model = SentenceTransformer("BAAI/bge-small-en-v1.5")

#The model wants a list 
print(f"Encoding {len(df_subset)} abstracts...")
abstract_list = df_subset['abstract_cleaned'].tolist()
embeddings = model.encode(abstract_list,show_progress_bar = True)
titles = [{"title": str(t)[:100]} for t in df_subset['article']]

client = chromadb.Client()

try:
    client.delete_collection(name = 'medical_docs')
except:
    pass
collection = client.create_collection(name = 'medical_docs')
collection.add(
    documents = abstract_list,
    embeddings = embeddings.tolist(),
    metadatas = titles,
    ids = [str(i) for i in range(len(df_subset))]
)

print(f"Done.{collection.count()} rows are now searchable ")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 1000 abstracts...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Done.1000 rows are now searchable 
